# Mundial 2026 en Colab gratuito (CPU)
Reutiliza modelos ya entrenados, fija resultados reales y guarda la caché en Drive. Usa `--force-retrain` solamente cuando quieras volver a ajustar el modelo.

In [ ]:
import io, os, pathlib, subprocess, sys, shutil, tempfile, zipfile
from google.colab import files
print('Selecciona Simulaciones_Mundial_Colab_CPU.zip')
subidos = files.upload()
nombre, contenido_zip = next(iter(subidos.items()))
destino = pathlib.Path(tempfile.mkdtemp(prefix='simulaciones_mundial_'))
with zipfile.ZipFile(io.BytesIO(contenido_zip)) as zf:
    for member in zf.infolist():
        nombre_limpio = member.filename.replace('\\', '/')
        objetivo = (destino / nombre_limpio).resolve()
        if destino.resolve() not in objetivo.parents and objetivo != destino.resolve():
            raise RuntimeError(f'Ruta insegura dentro del ZIP: {member.filename}')
        if member.is_dir() or nombre_limpio.endswith('/'):
            objetivo.mkdir(parents=True, exist_ok=True)
        else:
            objetivo.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as origen, open(objetivo, 'wb') as salida:
                shutil.copyfileobj(origen, salida)
candidatos = list(destino.rglob('requirements-colab.txt'))
if len(candidatos) != 1:
    raise RuntimeError(f'No se encontró una raíz única del proyecto: {candidatos}')
REPO = candidatos[0].parent
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-r', 'requirements-colab.txt'], check=True)
print('Repositorio y dependencias listos:', REPO)

In [ ]:
# Recomendado: reutilizar modelos entre sesiones de Colab.
USE_DRIVE = True
CACHE = pathlib.Path('/content/drive/MyDrive/Simulaciones_Mundial/modelos')
MODELOS = ['modelo_goles_L.pkl', 'modelo_goles_V.pkl', 'modelo_1X2_calibrado.pkl', 'columnas_entrenamiento.pkl']
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE.mkdir(parents=True, exist_ok=True)
    for nombre in MODELOS:
        origen = CACHE / nombre
        if origen.exists():
            shutil.copy2(origen, REPO / nombre)
print('Modelos recuperados:', [p for p in MODELOS if (REPO / p).exists()])

In [ ]:
# Agrega aqui solamente resultados nuevos; los anteriores se conservan.
import pandas as pd

columnas = ['Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante']
ruta_resultados = REPO / 'Data/resultados_reales.csv'
resultados = pd.read_csv(ruta_resultados) if ruta_resultados.exists() else pd.DataFrame(columns=columnas)

nuevos_resultados = pd.DataFrame([
    # Ejemplo despues de que termine:
    # {'Equipo_Local': 'Francia', 'Equipo_Visitante': 'Irak', 'Goles_Local': 2, 'Goles_Visitante': 0},
], columns=columnas)

if not nuevos_resultados.empty:
    resultados = pd.concat([resultados, nuevos_resultados], ignore_index=True)
    resultados.drop_duplicates(['Equipo_Local', 'Equipo_Visitante'], keep='last', inplace=True)

alias = {
    'Arabia Saudita': 'Arabia Saud?',
    'Bosnia y Herzegovina': 'Bosnia-Herzegovina',
    'Estados Unidos': 'EE. UU.',
    'Qatar': 'Catar',
}
resultados[['Equipo_Local', 'Equipo_Visitante']] = resultados[['Equipo_Local', 'Equipo_Visitante']].replace(alias)

calendario = pd.read_csv(REPO / 'Data/partidos_mundial.csv', usecols=['Equipo_Local', 'Equipo_Visitante'])
pares_validos = set(map(tuple, calendario.to_numpy()))
desconocidos = [tuple(x) for x in resultados[['Equipo_Local', 'Equipo_Visitante']].to_numpy() if tuple(x) not in pares_validos]
if desconocidos:
    raise ValueError(f'Partidos que no coinciden con el calendario: {desconocidos}')

resultados.to_csv(ruta_resultados, index=False)
print(f'Resultados acumulados y guardados: {len(resultados)}')
resultados.tail(10)


In [ ]:
# La primera ejecucion entrena; las siguientes reutilizan los .pkl.
subprocess.run([sys.executable, '04_Prediccion/prediccion_mundial.py',
                '--profile', 'colab', '--n-sims', '5000'], check=True)

In [ ]:
subprocess.run([sys.executable, '04_Prediccion/generar_informe.py'], check=True)
subprocess.run([sys.executable, '05_Web/generar_web.py'], check=True)
if USE_DRIVE:
    for nombre in MODELOS:
        shutil.copy2(REPO / nombre, CACHE / nombre)
print('Predicciones, informe y web generados.')

In [ ]:
salida = pathlib.Path('/content/resultados_mundial.zip')
with zipfile.ZipFile(salida, 'w', zipfile.ZIP_DEFLATED) as zf:
    for carpeta in ['Predicciones', 'docs']:
        for archivo in (REPO / carpeta).rglob('*'):
            if archivo.is_file():
                zf.write(archivo, archivo.relative_to(REPO).as_posix())
    zf.write(REPO / 'Data/resultados_reales.csv', 'Data/resultados_reales.csv')
files.download(str(salida))